In [1]:
import pandas as pd

train = pd.read_parquet('../data/processed/train.parquet')
val = pd.read_parquet('../data/processed/val.parquet')

print(train.shape, val.shape)

(9537, 17) (2385, 17)


In [2]:
train['text'] = train['subject'].fillna('') + ' ' + train['body'].fillna('')
val['text'] = val['subject'].fillna('') + ' ' + val['body'].fillna('')

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1, 2))

X_train_text = tfidf.fit_transform(train['text'])
X_val_text = tfidf.transform(val['text'])

In [4]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ['type', 'queue']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
X_train_cat = encoder.fit_transform(train[cat_cols])
X_val_cat = encoder.transform(val[cat_cols])

In [5]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

tag_cols = [f'tag_{i}' for i in range(1, 9)]

def get_tag_lists(df):
    return df[tag_cols].apply(lambda row: [t for t in row if pd.notna(t)], axis=1)

train_tag_lists = get_tag_lists(train)
val_tag_lists = get_tag_lists(val)

mlb = MultiLabelBinarizer()
train_tags_arr_full = mlb.fit_transform(train_tag_lists)

freq = np.asarray(train_tags_arr_full).sum(axis=0)
tag_freq = pd.Series(freq, index=mlb.classes_).sort_values(ascending=False)

top_n = 100
top_tags = tag_freq.head(top_n).index.tolist()

mlb_filtered = MultiLabelBinarizer(classes=top_tags)
train_tags_arr = mlb_filtered.fit_transform(train_tag_lists)
val_tags_arr = mlb_filtered.transform(val_tag_lists)

/Applications/support-ticket-mlops/.venv/lib/python3.11/site-packages/sklearn/preprocessing/_label.py:1016: UserWarning: unknown class(es) ['2019', 'AI', 'AWS', 'Access Control', 'Access Controls', 'Access-Control', 'AccessControls', 'AccessManagement', 'Accessibility', 'Accessory', 'Accounting', 'Accrual', 'Accuracy', 'Action', 'Activation', 'Ad', 'Ad Spending', 'AdBlocker', 'AdBlocking', 'AdContent', 'AdCreative', 'AdFatigue', 'AdPlacement', 'AdText', 'AddIns', 'Add_Ins', 'Address', 'Adjustment', 'Adobe', 'Adobe Premiere Pro', 'Adobe Sign', 'AdobeAudition', 'AdobeCreativeCloud', 'Advertising', 'Advice', 'Agency', 'Airtable', 'Algorithm', 'Algorithms', 'Alignment', 'Alternative', 'Alteryx', 'Amount', 'Analysis', 'Analysis,Troubleshooting', 'AnalyticsInconsistency', 'Android', 'Annual Plan', 'Ansible', 'Antivirus', 'Apache Hadoop', 'App', 'Application', 'Asana', 'Assessment', 'AssetRendering', 'Assurance', 'Audience', 'AudioSettings', 'Audit', 'Auditing', 'Audits', 'Authentication', 'A

In [6]:
from scipy.sparse import hstack, csr_matrix

X_train = hstack([X_train_text, X_train_cat, csr_matrix(train_tags_arr)])
X_val = hstack([X_val_text, X_val_cat, csr_matrix(val_tags_arr)])

y_train_priority = train['priority']
y_val_priority = val['priority']

y_train_escalated = train['escalated']
y_val_escalated = val['escalated']

print(X_train.shape, X_val.shape)

(9537, 2114) (2385, 2114)


In [7]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///../mlflow/mlflow.db")
mlflow.set_experiment("ticket-escalation-prediction")

2026/09/24 13:08:37 INFO mlflow.tracking.fluent: Experiment with name 'ticket-escalation-prediction' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Applications/support-ticket-mlops/notebooks/mlruns/1', creation_time=1790235517962, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1790235517962, lifecycle_stage='active', name='ticket-escalation-prediction', tags={}, trace_location=None, workspace='default'>

In [10]:
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression



with mlflow.start_run(run_name="baseline_logreg"):
    baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
    baseline.fit(X_train, y_train_escalated)
    preds = baseline.predict(X_val)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_metric("precision", precision_score(y_val_escalated, preds))
    mlflow.log_metric("recall", recall_score(y_val_escalated, preds))
    mlflow.log_metric("f1", f1_score(y_val_escalated, preds))

    mlflow.sklearn.log_model(baseline, "model")

2026/09/24 13:13:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [13]:

from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

scale_pos_weight = (y_train_escalated == 0).sum() / (y_train_escalated == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

with mlflow.start_run(run_name="xgboost_untuned_v1"):
    xgb_v1 = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    xgb_v1.fit(X_train, y_train_escalated)
    preds_v1 = xgb_v1.predict(X_val)
    probs_v1 = xgb_v1.predict_proba(X_val)[:, 1]

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)

    mlflow.log_metric("precision", precision_score(y_val_escalated, preds_v1))
    mlflow.log_metric("recall", recall_score(y_val_escalated, preds_v1))
    mlflow.log_metric("f1", f1_score(y_val_escalated, preds_v1))
    mlflow.log_metric("pr_auc", average_precision_score(y_val_escalated, probs_v1))
    mlflow.log_metric("roc_auc", roc_auc_score(y_val_escalated, probs_v1))

    mlflow.xgboost.log_model(xgb_v1, "model")

print("Untuned XGBoost run logged")

scale_pos_weight: 3.681885125184094


2026/09/24 13:24:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Untuned XGBoost run logged


In [ ]:
scale_pos_weight = (y_train_escalated == 0).sum() / (y_train_escalated == 1).sum()

with mlflow.start_run(run_name="xgboost_tuned_escalation"):
    xgb_model_v2 = XGBClassifier(
        n_estimators=400,       
        max_depth=8,           
        learning_rate=0.05,     
        scale_pos_weight=scale_pos_weight,
        eval_metric='logloss',
        random_state=42
    )
    xgb_model_v2.fit(X_train, y_train_escalated)
    preds = xgb_model_v2.predict(X_val)
    probs = xgb_model_v2.predict_proba(X_val)[:, 1]

    mlflow.log_param("model_type", "XGBoost")
    mlflow.log_param("n_estimators", 400)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("scale_pos_weight", scale_pos_weight)

    mlflow.log_metric("precision", precision_score(y_val_escalated, preds))
    mlflow.log_metric("recall", recall_score(y_val_escalated, preds))
    mlflow.log_metric("f1", f1_score(y_val_escalated, preds))
    mlflow.log_metric("pr_auc", average_precision_score(y_val_escalated, probs))
    mlflow.log_metric("roc_auc", roc_auc_score(y_val_escalated, probs))

    mlflow.xgboost.log_model(xgb_model_v2, "model")

print("XGBoost run logged")
print("PR-AUC:", average_precision_score(y_val_escalated, probs))

2026/09/24 13:25:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


XGBoost run logged
PR-AUC: 0.7633553944611469


In [15]:
runs = mlflow.search_runs(experiment_names=["ticket-escalation-prediction"])
print(runs[['run_id', 'tags.mlflow.runName', 'metrics.pr_auc']].sort_values('metrics.pr_auc', ascending=False))

                             run_id       tags.mlflow.runName  metrics.pr_auc
0  9457f238f9074ce79c850478b4af95ef  xgboost_tuned_escalation        0.763355
1  40503c9295024dbe82975ab776e55d68        xgboost_untuned_v1        0.732200
2  bdd6ab0f2e784258bc58da5c57d0c281           baseline_logreg             NaN


In [17]:
best_run_id = "9457f238f9074ce79c850478b4af95ef"

result = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="ticket-escalation-model"
)
print(result)

Registered model 'ticket-escalation-model' already exists. Creating a new version of this model...
2026/09/24 13:37:35 WARNING mlflow.tracking._model_registry.fluent: Run with id 9457f238f9074ce79c850478b4af95ef has no artifacts at artifact path 'model', registering model based on models:/m-27201bf4e2ff4427a315a1a3905dea6b instead


<ModelVersion: aliases=[], creation_timestamp=1790237255973, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1790237255973, metrics=None, model_id=None, name='ticket-escalation-model', params=None, run_id='9457f238f9074ce79c850478b4af95ef', run_link=None, source='models:/m-27201bf4e2ff4427a315a1a3905dea6b', status='READY', status_message=None, tags={}, user_id=None, version=1, workspace='default'>


Created version '1' of model 'ticket-escalation-model'.


In [18]:
import joblib
import os

os.makedirs('../src/serving/artifacts', exist_ok=True)

joblib.dump(tfidf, '../src/serving/artifacts/tfidf.pkl')
joblib.dump(encoder, '../src/serving/artifacts/encoder.pkl')
joblib.dump(mlb_filtered, '../src/serving/artifacts/tag_binarizer.pkl')
joblib.dump(top_tags, '../src/serving/artifacts/top_tags.pkl')

xgb_model_v2.save_model('../src/serving/artifacts/xgb_escalation_model.json')

print("All artifacts saved")

All artifacts saved
